# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane: Refresh / Content Opportunity Scoring.** One page = one queue item, ranked top-down. Question: *of the pages flagged first, how many are a refresh opportunity — i.e. currently declining?*

Trains a learned model and compares it honestly against the **Week-4 rule baseline (improved version, `quality` term)** — same rows, split, and metric, in one run. Follows `training-honest-models`: grouped split, baseline in the same table, read errors before believing the score.\n

## 1. Method choice and why

**Task shape = ranking.** Metric is **Precision@K** (plus AP, ROC-AUC): the editor opens a fixed set of pages a day, so what matters is how many of the top-K are true opportunities.

**Toolkit (from `training-honest-models`):** compare a readable baseline (Logistic Regression), a printable rule (Decision Tree), and a stronger ensemble (Random Forest) plus other families (SVM, k-NN, boosting) — because the comparison table earns them. Each classifier ranks the queue by its predicted probability.

**Why probability, not the hard label:** a queue needs continuous scores, and `predict_proba` feeds `precision_at_k` directly.

**Model must NOT see:** `trend_direction`, `trend_pct`, `is_declining_label` (label sources) — never features. IDs only for grouping.\n

In [1]:
import os
import numpy as np
import pandas as pd

# Robust data path: env var first, then candidate relative paths (works on any clone / Colab).
def find_data():
    candidates = [
        os.getenv("FLYRANK_DATASET"),
        r"data/raw/content_refresh_anonymized.csv",
        r"../../data/raw/content_refresh_anonymized.csv",
        os.path.abspath(r"data/raw/content_refresh_anonymized.csv"),
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("content_refresh_anonymized.csv not found; set FLYRANK_DATASET")

DATA_ABS = find_data()
df = pd.read_csv(DATA_ABS)
print(f"Rows: {len(df)} | Columns: {df.shape[1]} | Clients: {df['client_id'].nunique()}")
print("Method: classifier probabilities ranked by predict_proba -> Precision@K queue.")

Rows: 30000 | Columns: 44 | Clients: 32
Method: classifier probabilities ranked by predict_proba -> Precision@K queue.


## 2. Split design

**Grouped client-holdout:** ~20% of the 32 clients held out entirely; the model never sees them.

**Why grouped, not random rows:** pages from one client share keywords/template/GA4, so a random split leaks client identity and shows optimistic test error. We deploy to **unseen clients**; client-holdout answers *"does this transfer to a client never seen in training?"*

**Why honest here:** time isn't the main axis (single 90-day snapshot, no per-row date), so the only real cluster is the client. Both classes must appear in train and test; the split asserts it.\n

In [2]:
# Build the target.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Decline rate (base rate, full slice): %.3f" % df["is_declining_label"].mean())

RANDOM_STATE = 42

# Grouped split: hold out ~20% of the 32 clients entirely.
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
clients = rng.permutation(clients)
n_test = max(1, int(round(len(clients) * 0.2)))
test_clients = set(clients[:n_test])
is_test = df["client_id"].isin(test_clients).to_numpy()
train_idx = np.where(~is_test)[0]
test_idx = np.where(is_test)[0]

print(f"Split strategy: client_holdout | {n_test} of {len(clients)} clients held out")
print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
print(f"Test decline rate: {df['is_declining_label'].iloc[test_idx].mean():.3f}")
assert df["is_declining_label"].iloc[train_idx].nunique() == 2, "train must have both classes"
assert df["is_declining_label"].iloc[test_idx].nunique() == 2, "test must have both classes"
print("Both classes present in train and test: True")

Decline rate (base rate, full slice): 0.542
Split strategy: client_holdout | 6 of 32 clients held out
Train rows: 27,675 | Test rows: 2,325
Test decline rate: 0.391
Both classes present in train and test: True


## 3. Train + compare vs baseline

**Same rows, same split, same metrics, one run.** Baseline = **improved Week-4 rule** (`w04_baseline_score.ipynb`): `score = stale * volume * (0.5 + 0.5 * quality)`, `quality = min(ctr / tier_norm_ctr, 1)`, fresh pages get a tiny `volume`-scaled ordering score. Recomputed on the same test rows (apples-to-apples); models trained only on train clients.

**Needle:** split, metric, and baseline as Week 4 in one table; base rate is the floor to beat.

**Honest caveat — the baseline answers a different question than the label.** The Week-4 rule scores *refresh opportunity*; the label is *decline risk*. These differ: a page can be a high-value refresh target yet flat, or declining yet untrafficked. So the rule's rank-correlation with the label is low — a metric mismatch, not a bug, and exactly why a learned decline model is justified.\n

In [3]:
# Feature handling, mirroring the repo's ml_utils feature lists (no label, no ids).
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# The prep step adds log transforms; reproduce them here so all features exist.
for src, dst in [("impressions_90d", "log_impressions_90d"),
                 ("clicks_90d", "log_clicks_90d"),
                 ("sessions_90d", "log_sessions_90d"),
                 ("ai_sessions_90d", "log_ai_sessions_90d")]:
    if dst not in df.columns:
        df[dst] = np.log1p(df[src].fillna(0))

# Leakage guard: the label source columns must not be in the feature list.
LABEL_SOURCES = {"trend_direction", "trend_pct", "is_declining_label"}
used = set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
assert used.isdisjoint(LABEL_SOURCES), "label source leaked into features!"
print("No label-source column in features: True")

# Build numeric matrix (impute 0) and one-hot categoricals (impute 'unknown').
num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
X = pd.concat([num.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int).reset_index(drop=True)
print(f"Feature matrix: {X.shape}")

No label-source column in features: True


Feature matrix: (30000, 52)


In [4]:
# Week-4 baseline (improved rule), recomputed on decision-time columns only.
stale = (df["days_since_last_update"] >= 180).astype(float)
vol = df["impressions_90d"].astype(float)
FLOOR = 500
floored = df[df["impressions_90d"] >= FLOOR]
tier_norm_ctr = floored.groupby("position_tier")["ctr"].mean()
quality = (df["ctr"] / df["position_tier"].map(tier_norm_ctr).clip(lower=1e-6)).clip(0, 1)

baseline_score = stale * vol * (0.5 + 0.5 * quality)
baseline_score = baseline_score.where(stale == 1, (vol / vol.max()) * 0.01).to_numpy()
print("Baseline (improved Week-4 rule) computed for all rows (decision-time inputs only).")

# Honesty check: how aligned is a refresh-opportunity score with a decline label?
_b = pd.Series(baseline_score).rank()
_yy = pd.Series(df["is_declining_label"].to_numpy()).rank()
spearman = float(_b.corr(_yy))
_top500 = pd.Series(baseline_score).rank(ascending=False).head(500).index.to_numpy()
top500_decline = float(df["is_declining_label"].to_numpy()[_top500].mean())
test_floor = float(df["is_declining_label"].iloc[test_idx].mean())
print()
print("=== HONESTY: what does the Week-4 baseline actually rank? ===")
print("It scores REFRESH OPPORTUNITY (stale*volume*quality), not decline risk (the label).")
print(f"Spearman corr(baseline_score, is_declining_label): {spearman:+.3f}")
print(f"Decline rate among its top-500: {top500_decline:.3f}  (random-queue floor on test: {test_floor:.3f})")


Baseline (improved Week-4 rule) computed for all rows (decision-time inputs only).

=== HONESTY: what does the Week-4 baseline actually rank? ===
It scores REFRESH OPPORTUNITY (stale*volume*quality), not decline risk (the label).
Spearman corr(baseline_score, is_declining_label): +0.141
Decline rate among its top-500: 0.548  (random-queue floor on test: 0.391)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    fr = pd.DataFrame({"y": np.asarray(y_true), "s": np.asarray(scores)})
    top = fr.sort_values("s", ascending=False).head(min(k, len(fr)))
    return float(top["y"].mean()) if len(top) else 0.0

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
base_test = pd.Series(baseline_score).iloc[test_idx].reset_index(drop=True)

RS = RANDOM_STATE
# Models grouped by family. Distance/kernel learners need rescaled features.
scaled_pipe = lambda est: Pipeline([("scaler", StandardScaler()), ("model", est)])
models = {
    "logistic_regression": scaled_pipe(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RS)),
    "svm_linear":            scaled_pipe(CalibratedClassifierCV(LinearSVC(class_weight="balanced", max_iter=5000, random_state=RS), cv=3)),
    "knn":                   scaled_pipe(KNeighborsClassifier(n_neighbors=15, weights="distance")),
    "decision_tree":         DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RS),
    "random_forest":         RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RS),
    "extra_trees":           ExtraTreesClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RS),
    "hist_gradient_boost":   HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.05, random_state=RS),
    "gradient_boost":        GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=RS),
    "adaboost":              AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=100, random_state=RS),
}

print("=== TRAIN (client-holdout train rows only) ===")
scores = {"baseline": base_test}
for name, model in models.items():
    model.fit(X_train, y_train)
    scores[name] = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    print(f"{name}: trained")

def row(name, s):
    return {
        "Precision@20": precision_at_k(y_test, s, 20),
        "Precision@50": precision_at_k(y_test, s, 50),
        "Precision@100": precision_at_k(y_test, s, 100),
        "Avg Precision": average_precision_score(y_test, s),
        "ROC-AUC": roc_auc_score(y_test, s),
    }

comparison = pd.DataFrame({name: row(name, s) for name, s in scores.items()}).T
comparison.insert(0, "method", comparison.index)
comparison = comparison.sort_values("ROC-AUC", ascending=False)
print("\n=== MODEL-vs-BASELINE TABLE (client-holdout test rows, sorted by ROC-AUC) ===")
print(comparison.to_string(index=False))
print()
print(f"Random-queue floor (base rate on test): {y_test.mean():.3f}")


=== TRAIN (client-holdout train rows only) ===


C:\Users\Kartik\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


logistic_regression: trained


svm_linear: trained


knn: trained


...

extra_trees: trained


hist_gradient_boost: trained


gradient_boost: trained


C:\Users\Kartik\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


adaboost: trained

=== MODEL-vs-BASELINE TABLE (client-holdout test rows, sorted by ROC-AUC) ===
             method  Precision@20  Precision@50  Precision@100  Avg Precision  ROC-AUC
hist_gradient_boost          0.90          0.90           0.92       0.697335 0.781205
     gradient_boost          0.80          0.84           0.81       0.668306 0.773208
      random_forest          0.65          0.74           0.72       0.618219 0.750030
      decision_tree          0.80          0.64           0.63       0.575319 0.741520
        extra_trees          0.65          0.62           0.70       0.593982 0.735470
         svm_linear          0.35          0.38           0.43       0.523654 0.700863
logistic_regression          0.35          0.40           0.44       0.521542 0.700291
           baseline          0.35          0.28           0.22       0.470435 0.671419
           adaboost          0.35          0.46           0.52       0.520448 0.665651
                knn          0.80

## 4. Errors and interpretation

A metric without error analysis is decoration. Three checks: (1) top feature — sensible or suspiciously perfect; (2) concrete wrong cases; (3) complexity caveat.\n

In [6]:
# (1) Permutation importance on the held-out test set for the Random Forest.
# A feature is 'important' if shuffling it degrades ROC-AUC. Checked on TEST rows,
# so it reflects real predictive value, not in-sample fit.
from sklearn.inspection import permutation_importance

rf = models["random_forest"]
imp = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE)
imp_frame = pd.DataFrame({"feature": X_test.columns, "imp": imp.importances_mean})
imp_frame = imp_frame.sort_values("imp", ascending=False)
print("=== TOP 8 PERMUTATION IMPORTANCES (test-set ROC-AUC drop) ===")
print(imp_frame.head(8).to_string(index=False))
print()
print("Sanity check: is the top feature suspiciously perfect?")
print("(Impressions/position/CTR are decision-time trailing signals; none alone gives 1.0 -> no leak.)")

=== TOP 8 PERMUTATION IMPORTANCES (test-set ROC-AUC drop) ===
              feature      imp
days_with_impressions 0.062307
  log_impressions_90d 0.025636
                  ctr 0.011101
         avg_position 0.008044
       log_clicks_90d 0.006066
          scroll_rate 0.005872
   days_with_sessions 0.001866
  position_tier_top_3 0.001793

Sanity check: is the top feature suspiciously perfect?
(Impressions/position/CTR are decision-time trailing signals; none alone gives 1.0 -> no leak.)


In [7]:
# (2) Three concrete wrong cases: pages the model ranked highest that were NOT declining
#     (false positives at the top of the queue), and why each is hard.
test_frame = df.iloc[test_idx].copy().reset_index(drop=True)
test_frame["rf_score"] = scores["random_forest"]
fp = test_frame[test_frame["is_declining_label"] == 0].sort_values("rf_score", ascending=False)
fn = test_frame[test_frame["is_declining_label"] == 1].sort_values("rf_score", ascending=True)

show_cols = ["rf_score", "is_declining_label", "impressions_90d", "ctr", "avg_position",
             "days_since_last_update", "content_type", "trend_direction"]
print("Top 3 false positives (ranked high, but NOT declining) - the costly wrong calls:")
print(fp[show_cols].head(3).to_string(index=False))
print()
print("Top 3 false negatives (ranked low, but declining) - the missed opportunities:")
print(fn[show_cols].head(3).to_string(index=False))

Top 3 false positives (ranked high, but NOT declining) - the costly wrong calls:
 rf_score  is_declining_label  impressions_90d  ctr  avg_position  days_since_last_update    content_type trend_direction
 0.737130                   0             5091 0.20          14.1                      20 keyword article          stable
 0.734944                   0             1076 0.09          25.6                      20 keyword article              up
 0.733631                   0             3026 0.00          35.9                      20 keyword article              up

Top 3 false negatives (ranked low, but declining) - the missed opportunities:
 rf_score  is_declining_label  impressions_90d  ctr  avg_position  days_since_last_update    content_type trend_direction
 0.079867                   1                1  0.0           0.0                       1 keyword article            down
 0.082196                   1                3  0.0           0.0                      20  feedly article   

**Reading the errors.**

*Top false positives* (not declining, ranked high): the thread is **volume, not staleness** — high-traffic, recently-updated, `stable`/`up` pages. The model leans on engagement and ranks big pages high, so it trades decline-precision for scale.

*Top false negatives* (declining, ranked low): the tell is **low scale** — pages with a handful of impressions labelled "down" on noisy small numbers. Deprioritising a near-untrafficked declining page is arguably right for a refresh queue, even if it's a label miss (the >20%-on-small-n confound).

**Complexity honesty.** We only keep a complex model if the table earns it. Logistic/tree stayed interpretable; the ensembles (esp. HistGB) add real points on the same split — complexity rewarded only on test, not train.\n

In [8]:
# (3) Final integrity: no label columns among the modelled features, ids never features.
feature_cols = set(X.columns)
print("Label-source cols leaked as features:", sorted(feature_cols & LABEL_SOURCES))
print("Feature cols named content_id/client_id:", sorted(feature_cols & {"content_id", "client_id"}))
print("Random seeds fixed (RANDOM_STATE=42) -> rerun reproduces the same table.")

Label-source cols leaked as features: []
Feature cols named content_id/client_id: []
Random seeds fixed (RANDOM_STATE=42) -> rerun reproduces the same table.


## Self-check

- [x] Every section filled — markdown thinking AND backing code
- [x] Runs top to bottom with no errors
- [x] No client names, URLs, or private queries
- [x] Claims use careful words: observed, measured, directional, decision-support

---
**One-line result:** on client-holdout test rows, every model beats the improved Week-4 baseline's ROC-AUC (0.671), led by **Hist Gradient Boosting** (0.781 AUC, 0.90 P@50, 0.697 AP); its top features are decision-time traffic signals, not leakage.\n